# ZIP-RC: Colab runtime from VS Code

This notebook runs a small but real end-to-end ZIP-RC training pipeline. Each heading is a foldable section in VS Code. Run the sections in order.

The safe profile uses Qwen3-0.6B on GPUs below 35 GB and the paper's Qwen3-1.7B on an A100-class runtime. The released repository does not include the paper's adaptive meta-action sampler.

## 1. Check the Colab GPU

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)

## 2. Install the authors' pinned environment

This intentionally omits `flash-attn`: the training code falls back to standard attention. Installation can take 5-15 minutes. If VS Code asks to restart the kernel afterward, restart it and continue from section 3.

In [ ]:
import subprocess, sys
packages = [
    "torch==2.6.0",
    "vllm==0.8.5.post1",
    "transformers==4.51.3",
    "datasets==3.5.0",
    "accelerate==1.6.0",
    "pandas==2.2.3",
    "pyarrow==19.0.1",
    "sentencepiece==0.2.0",
    "matplotlib==3.10.3",
    "wandb==0.19.10",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

## 3. Clone upstream and choose a memory-safe profile

In [ ]:
from pathlib import Path
import subprocess, torch

REPO = Path("/content/ZIP-RC")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/rohinmanvi/ZIP-RC.git", str(REPO)], check=True)

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
MODEL_ID = "Qwen/Qwen3-1.7B" if VRAM_GB >= 35 else "Qwen/Qwen3-0.6B"
MAX_LENGTH = 2048 if VRAM_GB >= 35 else 1024
PROMPTS = 8
TRAIN_STEPS = 10 if VRAM_GB >= 35 else 4

DATA = REPO / "data/colab_rollouts.parquet"
VALUE_DATA = REPO / "data/colab_rollouts_with_value.parquet"
INTERMEDIATE = REPO / "models/colab_joint_correct"
FINAL = REPO / "models/colab_ziprc_final"
(REPO / "data").mkdir(exist_ok=True)
(REPO / "models").mkdir(exist_ok=True)

print({
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(VRAM_GB, 1),
    "model": MODEL_ID,
    "max_length": MAX_LENGTH,
    "training_steps_per_stage": TRAIN_STEPS,
})

In [ ]:
def run_repo(*args):
    command = [str(x) for x in args]
    print("Running:", " ".join(command))
    return subprocess.run(command, cwd=REPO, check=True)

## 4. Generate on-policy rollouts

In [ ]:
run_repo(
    "python3", "src/generate_ziprc_rollouts.py",
    "--model", MODEL_ID,
    "--dataset", "rohinm/adaptivemath",
    "--split", "train",
    "--prompt-column", "problem",
    "--answer-column", "answer",
    "--out", DATA,
    "--max-num-prompts", PROMPTS,
    "--thinking-samples", 0,
    "--non-thinking-samples", 1,
    "--temperature", 1.0,
    "--min-p", 0.1,
    "--max-model-len", MAX_LENGTH,
    "--max-num-seqs", 2,
    "--dp-size", 1,
    "--tp-size", 1,
)

## 5. Grade the rollouts

The upstream smoke test uses a 30B grader over eight GPUs. This single-GPU notebook deliberately uses the selected small model as the grader; it validates the code path but produces noisier correctness labels.

In [ ]:
run_repo(
    "python3", "src/evaluate_and_label_rollouts.py",
    "--data", DATA,
    "--model", MODEL_ID,
    "--tensor-parallel-size", 1,
    "--gpu-memory-utilization", 0.85,
    "--max-model-len", MAX_LENGTH,
    "--max-num-seqs", 2,
)

## 6. Train the intermediate reward-length head

Gradient accumulation is explicitly 1. With the upstream smoke defaults (8 rows, accumulation 16, and 2 epochs), the optimizer may never step.

In [ ]:
run_repo(
    "python3", "src/train_ziprc_joint_head.py",
    "--model-id", MODEL_ID,
    "--data-path", DATA,
    "--weights-path", INTERMEDIATE,
    "--distribution-token-id", 151669,
    "--label-column", "correct",
    "--kl-coefficient", 0.0,
    "--batch-size", 1,
    "--gradient-accumulation-steps", 1,
    "--num-epochs", 3,
    "--max-steps", TRAIN_STEPS,
    "--max-length", MAX_LENGTH,
)

## 7. Write denoised value labels

In [ ]:
run_repo(
    "python3", "src/score_with_ziprc_joint_head.py",
    "--model", INTERMEDIATE,
    "--in-parquet", DATA,
    "--out-parquet", VALUE_DATA,
    "--distribution-token-id", 151669,
    "--num-length-bins", 8,
    "--reward-values", 0.0, 1.0,
    "--last-k", 64,
    "--max-length", MAX_LENGTH,
    "--num-workers", 0,
)

## 8. Train the final KL-regularized ZIP-RC model

This is the peak-memory stage because it loads both the student and frozen reference model. If it runs out of memory, reconnect to a larger GPU or set `MODEL_ID` to Qwen3-0.6B and rerun from section 4.

In [ ]:
run_repo(
    "python3", "src/train_ziprc_joint_head.py",
    "--model-id", MODEL_ID,
    "--data-path", VALUE_DATA,
    "--weights-path", FINAL,
    "--distribution-token-id", 151669,
    "--label-column", "value",
    "--kl-coefficient", 10.0,
    "--batch-size", 1,
    "--gradient-accumulation-steps", 1,
    "--num-epochs", 3,
    "--max-steps", TRAIN_STEPS,
    "--max-length", MAX_LENGTH,
)

## 9. Inspect and save results

In [ ]:
import pandas as pd
df = pd.read_parquet(VALUE_DATA)
display(df[["prompt", "length", "finished", "correct", "value"]].head(8))
print("Final model files:", [p.name for p in FINAL.iterdir()])

In [ ]:
# First run `Colab: Mount Google Drive to Server...` from the VS Code command palette.
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    destination = drive_root / "ZIP-RC-results"
    destination.mkdir(exist_ok=True)
    shutil.copytree(FINAL, destination / FINAL.name, dirs_exist_ok=True)
    shutil.copy2(VALUE_DATA, destination / VALUE_DATA.name)
    print("Saved to", destination)
else:
    print("Drive is not mounted. Results remain in ephemeral storage:", FINAL)